In [ ]:
import tensorflow as tf
import os

#limit memory growth but might not need
gpus = tf.config.experimental.list_physical_devices('CPU')
for gpus in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
#LOAD DATA
#1. Dependencies

import numpy as np
from matplotlib import pyplot as plt
import cv2

In [ ]:
#2. loading the data
data = tf.keras.utils.image_dataset_from_directory('data', batch_size=32)          #builds dataset on the fly, batch sizes into 32, image size 256. Folder called data

In [ ]:
data_iterator = data.as_numpy_iterator()        #ALLOWS access to the pipeline

In [ ]:
batch = data_iterator.next()        #actually accessing it. batch = bach of data. Gets another batch from iterator

In [ ]:
#PREPROCESSING DATA
#Scaling
data = data.map(lambda x,y: (x/255,y))           #data.map allows transformation in python. x = images. y = target label. we will scale by 255
scaled_iterator = data.as_numpy_iterator().next()                     #grabs next batch
batch = scaled_iterator.next()

In [ ]:
#SPLIT DATA : Train and Test partition
train_size = int(len(data)*.7)      #70% of training
val_size = int(len(data)*.2)+1        #20% validation
test_size = int(len(data)*.1)+1

In [ ]:
#take and skip method available in tensorflow
#take = how much data we are going to take in partition
train = data.take(train_size)
val = data.skip(train_size).take(val_size)      #skips batches we already allocated into batches
test = data.skip(train_size + val_size).take(test_size)

START MODELING!! Keras Sequential API

In [ ]:
#1. Build Deep learning Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Dropout, Flatten, Dense      
#Conv2D is 2D convolutional layer, uses max pooling

In [ ]:
model = Sequential()

#4 convolution blocs
model.add(Conv2D(32, (3,3), 2, activation='relu', padding='same', input_shape=(256,256,13)))     #1st layer needs input : model.add(layer, 16 filters, scan size, moves 1, activation function = relu)
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(64, (3,3), 2, activation='relu', padding='same'))              #i incorporated padding to retain more border informations
model.add(BatchNormalization()) 
model.add(MaxPooling2D())   

model.add(Conv2D(128, (3,3), 2, activation='relu', padding='same'))
model.add(BatchNormalization())
model.add(MaxPooling2D())

model.add(Conv2D(256, (3,3), 2, activation='relu', padding='same', name="last_conv"))           #Incorporate GRADCAM on the last layer
model.add(BatchNormalization())
model.add(MaxPooling2D())
#Flatten layer for the dense
model.add(Flatten())  #turning it into a single dimension.

#dense layer
model.add(Dense(256, activation='relu'))
model.add(Dense(3, activation='softmax'))           #3 tillage classes

In [ ]:
model.compile('adam', loss=tf.losses.BinaryCrossentropy(), metrics=['accuracy'])        #can pass more metrics

In [ ]:
model.summary()

In [ ]:
#2. Start Training
logdir='logs'
tensorboard_callback = tf.keras.callbacks.TensoBoard(log_dir = logdir)          #calls/save model training while it trains
hist = model.fit(train, epochs=20, validation_data=val, callbacks=[tensorboard_callback])                  #fit is the training component, predict is when we actually predict, epochs = 1 run entire data

In [ ]:
# GRADCAM Visualization

def make_gradcam_heatmap(model, images, last_conv_layer_name, class_index=None):
    """
    images: tensor of shape (1, 256, 256, 3)
    """

    grad_model = tf.keras.models.Model(
        [model.input],
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(images)
        if class_index is None:
            class_index = tf.argmax(predictions[0])
        loss = predictions[:, class_index]

    grads = tape.gradient(loss, conv_outputs)

    # Global average pooling
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    # Normalize
    heatmap = tf.maximum(heatmap, 0)
    heatmap /= tf.reduce_max(heatmap)

    return heatmap.numpy()


In [ ]:
#3. Plot Performance

#plotting loss
fig = plt.figure()
plt.plot(hist.history['loss'], color='teal', label='loss')              #plot training loss
plt.plot(hist.history['val_loss'], color='orange', label='val_loss')          #plot validation loss
fig.suptitle('Loss', fontsize=20)
plt.legend(loc="upper left")
plt.show()

In [ ]:
#Plotting Accuracy
fig = plt.figure()
plt.plot(hist.history['accuracy'], color='teal', label='accuracy')              #plot training loss
plt.plot(hist.history['val_accuracy'], color='orange', label='accuracy')          #plot validation loss
fig.suptitle('Accuracy', fontsize=20)
plt.legend(loc="upper left")
plt.show()

In [ ]:
#pulling out one image to test
for images, labels in test.take(1):
    img = images[0:1]   # shape (1, 256, 256, 3)
    label = labels[0]

In [ ]:
#finally generating the image with gradcam
heatmap = make_gradcam_heatmap(
    model,
    img,
    last_conv_layer_name="last_conv"
)


In [ ]:
#final image
img_np = img[0].numpy()

heatmap = cv2.resize(heatmap, (256,256))
heatmap = np.uint8(255 * heatmap)

heatmap_color = cv2.applyColorMap(heatmap, cv2.COLORMAP_JET)

superimposed = cv2.addWeighted(
    np.uint8(img_np * 255),
    0.6,
    heatmap_color,
    0.4,
    0
)

plt.figure(figsize=(6,6))
plt.imshow(cv2.cvtColor(superimposed, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()
